# Trabajo individual — Spotify, The Smiths y letras

Este notebook está preparado para ejecutarse directamente en Google Colab sin clonar repositorios ni depender de archivos externos del proyecto.

## Objetivos del trabajo

1. Filtrar los álbumes de The Smiths para conservar solo álbumes de estudio.
2. Crear gráficos relacionados con letras de canciones usando `lyrics.ovh`.
3. Indicar la web app Streamlit desarrollada para explorar los componentes.

## 1. Instalación de librerías

Esta instalación se realiza solo dentro del entorno temporal de Colab. No instala nada en el ordenador.

In [ ]:
!pip install -q gdown wordcloud requests pandas matplotlib

## 2. Carga del dataset

Se utiliza el fichero `tracks_features.csv` del dataset Spotify 1.2M+ Songs.  
El notebook lo descarga automáticamente usando el mismo identificador de Google Drive que aparecía en el ejemplo base.

In [ ]:
import os
import re
import ast
import time
from collections import Counter

import gdown
import pandas as pd
import matplotlib.pyplot as plt
import requests
from wordcloud import WordCloud
from IPython.display import display

DATA_FILE = "tracks_features.csv"
GDRIVE_FILE_ID = "1jsXTNtGhOrsCApQctYx-hRxAQASAcPlI"

if not os.path.exists(DATA_FILE):
    print("Descargando dataset...")
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", DATA_FILE, quiet=False)
else:
    print("Dataset ya existe en Colab.")

df = pd.read_csv(DATA_FILE)

print("Dimensiones:", df.shape)
print("Columnas:")
print(df.columns.tolist())

display(df.head())

## 3. Filtrado exacto del artista

Inicialmente se comprobó que buscar `"The Smiths"` con `str.contains()` generaba falsos positivos, por ejemplo `The Smithsonian Chamber Players`.  
Por eso aquí se usa un filtro exacto sobre la lista de artistas.

In [ ]:
TARGET_ARTIST_NAME = "The Smiths"

def parse_possible_list(value):
    if pd.isna(value):
        return []
    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed]
        return [str(parsed).strip()]
    except Exception:
        return [text.strip()]

def artist_exact_match(artists_value, target_artist):
    artists = parse_possible_list(artists_value)
    return any(a.lower().strip() == target_artist.lower() for a in artists)

contains_mask = df["artists"].astype(str).str.contains(TARGET_ARTIST_NAME, case=False, na=False)
exact_mask = df["artists"].apply(lambda x: artist_exact_match(x, TARGET_ARTIST_NAME))

contains_df = df[contains_mask].copy()
smiths_df = df[exact_mask].copy()

print(f"Filas encontradas con str.contains('The Smiths'): {len(contains_df)}")
print(f"Filas encontradas con artista exacto 'The Smiths': {len(smiths_df)}")
print(f"Posibles falsos positivos: {len(contains_df) - len(smiths_df)}")

display(smiths_df[["name", "album", "artists", "year"]].sort_values(["year", "album", "name"]))

## 4. Álbumes encontrados antes del filtrado

Esta tabla muestra qué publicaciones aparecen en el dataset para The Smiths como artista exacto.

In [ ]:
albums_before = (
    smiths_df
    .groupby("album")
    .agg(canciones=("name", "count"), year=("year", "min"))
    .sort_values(["year", "album"])
)

display(albums_before)

## 5. Filtrado de álbumes de estudio

Según la discografía oficial de The Smiths, sus álbumes de estudio son:

- The Smiths
- Meat Is Murder
- The Queen Is Dead
- Strangeways, Here We Come

Al aplicar el filtro al dataset usado, solo aparecen canciones originales del álbum `Strangeways, Here We Come`.  
Esto no significa que el filtro esté mal, sino que el dataset no contiene los otros tres álbumes interpretados por The Smiths.

In [ ]:
def normalize_album_name(album):
    album = str(album)
    album = re.sub(r"\(.*?\)", "", album)
    album = re.sub(r"\[.*?\]", "", album)
    album = re.sub(r"\s*-\s*.*Remaster.*$", "", album, flags=re.IGNORECASE)
    album = re.sub(r"\s*-\s*.*Deluxe.*$", "", album, flags=re.IGNORECASE)
    album = re.sub(r"\s*-\s*.*Edition.*$", "", album, flags=re.IGNORECASE)
    album = re.sub(r"\s+", " ", album).strip()
    return album

STUDIO_ALBUMS = [
    "The Smiths",
    "Meat Is Murder",
    "The Queen Is Dead",
    "Strangeways, Here We Come"
]

smiths_df["short_album_name"] = smiths_df["album"].apply(normalize_album_name)

studio_df = smiths_df[smiths_df["short_album_name"].isin(STUDIO_ALBUMS)].copy()

studio_df = (
    studio_df
    .sort_values(["year", "short_album_name", "track_number", "name"])
    .drop_duplicates(subset=["short_album_name", "name"], keep="first")
    .reset_index(drop=True)
)

album_counts = []
for album in STUDIO_ALBUMS:
    count = int((studio_df["short_album_name"] == album).sum())
    album_counts.append({"album": album, "canciones": count})

album_counts_df = pd.DataFrame(album_counts)

print(f"Total de canciones después de filtrar álbumes de estudio: {len(studio_df)}")
display(album_counts_df)

display(studio_df[["track_number", "name", "short_album_name", "album", "year"]])

### Comentario

Como tres de los cuatro álbumes de estudio no tienen canciones disponibles en este dataset, los gráficos principales se hacen **por canción**, no por álbum.  
Esto permite obtener visualizaciones más útiles con las canciones realmente disponibles.

In [ ]:
studio_df["clean_name"] = (
    studio_df["name"]
    .astype(str)
    .str.replace(r"\s*-\s*2011 Remaster.*$", "", regex=True)
    .str.replace(r"\s*-\s*Remaster.*$", "", regex=True)
)

studio_df_plot = studio_df.sort_values("track_number").copy()

display(studio_df_plot[["track_number", "clean_name", "energy", "valence", "danceability", "acousticness", "tempo", "duration_ms"]])

## 6. Gráfico: Energy por canción

`energy` representa la intensidad percibida de una canción.

In [ ]:
plt.figure(figsize=(11, 6))
plt.barh(studio_df_plot["clean_name"], studio_df_plot["energy"])
plt.xlabel("Energy")
plt.ylabel("Canción")
plt.title("Energy por canción — The Smiths")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Gráfico: Valence por canción

`valence` mide la positividad musical. Valores bajos suelen asociarse con canciones más oscuras o melancólicas.

In [ ]:
plt.figure(figsize=(11, 6))
plt.barh(studio_df_plot["clean_name"], studio_df_plot["valence"])
plt.xlabel("Valence / positividad musical")
plt.ylabel("Canción")
plt.title("Valence por canción — The Smiths")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Gráfico: Energy vs Valence

Este gráfico permite comparar intensidad y positividad musical en las canciones disponibles.

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(studio_df_plot["valence"], studio_df_plot["energy"], alpha=0.8)

for _, row in studio_df_plot.iterrows():
    plt.text(row["valence"], row["energy"], row["clean_name"][:22], fontsize=8, alpha=0.8)

plt.xlabel("Valence / positividad musical")
plt.ylabel("Energy / energía")
plt.title("Energy vs Valence por canción — The Smiths")
plt.tight_layout()
plt.show()

## 9. Análisis de letras con lyrics.ovh

Se intentó obtener letras de The Smiths con `lyrics.ovh`, pero durante las pruebas la API no devolvió resultados para esas canciones.  
Como el enunciado permite analizar letras de un artista **o de un conjunto de artistas**, se usa un conjunto alternativo de canciones conocidas.

In [ ]:
def clean_track_title_for_lyrics(title):
    title = str(title)
    title = re.sub(r"\s*-\s*\d{4}\s*Remaster.*$", "", title, flags=re.IGNORECASE)
    title = re.sub(r"\s*-\s*Remaster.*$", "", title, flags=re.IGNORECASE)
    title = re.sub(r"\s*\(.*?\)", "", title)
    title = re.sub(r"\s*\[.*?\]", "", title)
    title = re.sub(r"\s+", " ", title)
    return title.strip()

def get_lyrics_lyricsovh(artist, title):
    clean_title = clean_track_title_for_lyrics(title)
    url = f"https://api.lyrics.ovh/v1/{requests.utils.quote(artist)}/{requests.utils.quote(clean_title)}"
    try:
        response = requests.get(url, timeout=12)
        if response.status_code == 200:
            data = response.json()
            lyrics = data.get("lyrics")
            if lyrics and len(lyrics.strip()) > 0:
                return lyrics
        return None
    except Exception:
        return None

# Intento inicial con The Smiths
lyrics_rows = []

for _, row in studio_df.head(10).iterrows():
    title = row["clean_name"]
    lyrics = get_lyrics_lyricsovh("The Smiths", title)
    if lyrics:
        lyrics_rows.append({
            "artist": "The Smiths",
            "song": title,
            "source": "The Smiths",
            "lyrics": lyrics,
            "lyrics_length": len(lyrics.split())
        })
    time.sleep(0.2)

lyrics_df = pd.DataFrame(lyrics_rows)
print(f"Letras de The Smiths encontradas: {len(lyrics_df)}")

## 10. Conjunto alternativo de artistas para las letras

In [ ]:
if lyrics_df.empty:
    lyrics_candidates = [
        {"artist": "Coldplay", "song": "Yellow"},
        {"artist": "Coldplay", "song": "The Scientist"},
        {"artist": "Radiohead", "song": "Creep"},
        {"artist": "Oasis", "song": "Wonderwall"},
        {"artist": "Queen", "song": "Bohemian Rhapsody"},
        {"artist": "Nirvana", "song": "Smells Like Teen Spirit"},
        {"artist": "Adele", "song": "Hello"},
        {"artist": "The Beatles", "song": "Hey Jude"},
        {"artist": "R.E.M.", "song": "Losing My Religion"},
        {"artist": "David Bowie", "song": "Heroes"},
    ]

    alt_rows = []
    for item in lyrics_candidates:
        artist = item["artist"]
        song = item["song"]
        lyrics = get_lyrics_lyricsovh(artist, song)
        if lyrics:
            alt_rows.append({
                "artist": artist,
                "song": song,
                "source": "Conjunto de artistas",
                "lyrics": lyrics,
                "lyrics_length": len(lyrics.split())
            })
        time.sleep(0.2)

    lyrics_df = pd.DataFrame(alt_rows)

print(f"Letras encontradas para el análisis textual: {len(lyrics_df)}")
display(lyrics_df[["artist", "song", "lyrics_length"]])

## 11. Gráfico: palabras más frecuentes

Se eliminan palabras muy comunes para visualizar mejor el vocabulario dominante en las letras recuperadas.

In [ ]:
STOPWORDS = {
    "the", "and", "you", "your", "that", "this", "with", "for", "are",
    "was", "were", "but", "not", "have", "has", "had", "they", "them",
    "his", "her", "she", "him", "our", "out", "all", "can", "will",
    "just", "from", "there", "what", "when", "where", "who", "why",
    "how", "into", "about", "over", "under", "then", "than", "too",
    "i", "me", "my", "we", "us", "a", "an", "of", "to", "in", "on",
    "it", "is", "be", "am", "do", "so", "no", "yes", "oh", "la",
    "na", "yeah", "hey", "got", "get", "let"
}

def tokenize_lyrics(text):
    text = str(text).lower()
    words = re.findall(r"[a-zA-Z']+", text)
    return [w for w in words if w not in STOPWORDS and len(w) > 2]

all_words = []
for lyrics in lyrics_df["lyrics"]:
    all_words.extend(tokenize_lyrics(lyrics))

word_counts = Counter(all_words)
words_df = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])

display(words_df)

plt.figure(figsize=(10, 6))
plt.barh(words_df["word"], words_df["frequency"])
plt.xlabel("Frecuencia")
plt.ylabel("Palabra")
plt.title("Palabras más frecuentes en las letras analizadas")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Nube de palabras

La nube de palabras ayuda a identificar visualmente las palabras más frecuentes.

In [ ]:
text_for_wordcloud = " ".join(lyrics_df["lyrics"].astype(str))

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    stopwords=STOPWORDS,
    max_words=100
).generate(text_for_wordcloud)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Nube de palabras de las letras analizadas")
plt.tight_layout()
plt.show()

## 13. Web app Streamlit

Además del notebook, se ha creado una web app con Streamlit para explorar los componentes implementados.

https://5mbkvc4sqvcmbqokwmnsou.streamlit.app/

La app permite explorar canciones, variables musicales, análisis de letras y gráficos.

## 14. Conclusiones

El proyecto cumple los tres apartados del enunciado:

1. Se filtraron los álbumes de estudio de The Smiths usando una lista basada en su discografía oficial.
2. Se incorporaron gráficos relacionados con letras: frecuencia de palabras y nube de palabras.
3. Se creó una web app en Streamlit para explorar los resultados.

Una conclusión importante es que el dataset usado solo contiene canciones originales de The Smiths del álbum `Strangeways, Here We Come`. Por eso, los gráficos musicales se realizaron por canción y no por álbum.